In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

from dataloader import samlet_df
from dataloader import (
    skjorte_data, shorts_data, bukse_data, tshirt_data,
    langærmet_data, jakke_data, fleece_data, overall_data,
    forklæde_data, kittel_data, busseron_data, kokkejakke_data,
    andre_data
)

# Read in samlet_alle
samlet_alle = pd.read_csv(
    "/Users/mathias/Documents/GitHub/Dataprojekt/2DHistogram/Saved Histograms/Samlet_Alle_20260323_105426.csv"
)

# ── Helper: compute percentage distribution of a categorical column ──────────
def get_pct(df: pd.DataFrame, col: str) -> pd.Series:
    """Return value-counts normalised to 100 % for a given column."""
    counts = df[col].value_counts(dropna=False)
    return (counts / counts.sum() * 100).rename(col)


# ── Choose which columns to visualise ────────────────────────────────────────
# Use all columns that exist in both DataFrames (excluding obvious id/index cols)
exclude_cols = {"index", "Unnamed: 0"}
shared_cols = [
    c for c in samlet_alle.columns
    if c in samlet_df.columns and c not in exclude_cols
]

if not shared_cols:
    raise ValueError(
        "No shared columns found between samlet_alle and samlet_df. "
        "Check column names."
    )

print(f"Plotting {len(shared_cols)} shared column(s): {shared_cols}")

# ── Build one plot per shared column ─────────────────────────────────────────
for col in shared_cols:
    pct_alle = get_pct(samlet_alle, col)
    pct_df   = get_pct(samlet_df,   col)

    # Align on the union of categories, fill missing with 0
    all_cats = sorted(set(pct_alle.index) | set(pct_df.index), key=str)
    pct_alle = pct_alle.reindex(all_cats, fill_value=0)
    pct_df   = pct_df.reindex(all_cats,   fill_value=0)

    n_cats   = len(all_cats)
    x        = np.arange(n_cats)          # one cluster per category
    bar_w    = 0.35                        # width of each bar
    cmap     = plt.get_cmap("tab10")

    fig, ax = plt.subplots(figsize=(max(8, n_cats * 1.2), 6))

    # Draw stacked bars for samlet_alle (left bar of each cluster)
    bottom_alle = np.zeros(n_cats)
    for i, cat in enumerate(all_cats):
        ax.bar(
            x - bar_w / 2,
            pct_alle.values,
            bar_w,
            bottom=bottom_alle,
            color=cmap(i % 10),
            label=str(cat) if i == 0 else "_nolegend_",
        )
        bottom_alle += pct_alle.values  # only one bar per category here,
        break                           # so the "stack" is the full bar

    # ── Proper stacked bars ───────────────────────────────────────────────────
    # Each bar = one category, height = its % share → naturally "stacked" view
    # Left  bar  = samlet_alle distribution
    # Right bar  = samlet_df   distribution

    # Reset and draw properly
    ax.cla()

    colors = [cmap(i % 10) for i in range(n_cats)]

    # samlet_alle  – each segment is one category value
    bottom = np.zeros(2)  # two bars: index 0 = alle, index 1 = df
    for i, cat in enumerate(all_cats):
        heights = np.array([pct_alle[cat], pct_df[cat]])
        bars = ax.bar(
            [0, 1],
            heights,
            width=0.5,
            bottom=bottom,
            color=colors[i],
            label=str(cat),
            edgecolor="white",
            linewidth=0.6,
        )
        # Annotate segments that are large enough to read
        for j, (b, h) in enumerate(zip(bottom, heights)):
            if h >= 3:
                ax.text(
                    j, b + h / 2,
                    f"{h:.1f}%",
                    ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold"
                )
        bottom += heights

    # ── Cosmetics ─────────────────────────────────────────────────────────────
    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        [f"samlet_alle\n(n={len(samlet_alle):,})",
         f"samlet_df\n(n={len(samlet_df):,})"],
        fontsize=11
    )
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 105)
    ax.set_ylabel("Percentage (%)", fontsize=11)
    ax.set_title(f"Distribution comparison – '{col}'", fontsize=13, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

    # Legend outside the plot
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels,
        title=col,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=9,
    )

    plt.tight_layout()
    safe_col = col.replace("/", "_").replace(" ", "_")
    out_path = f"stacked_bar_{safe_col}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved → {out_path}")
    plt.show()

Plotting 9 shared column(s): ['Produkt - Produkt', 'Kassationsårsag (ui)', 'Stk. tøj per kassationsdato', 'Dage i cirkulation', 'Total antal vask', 'Unik Kode (ui)', 'Måned', 'Vask per måned', 'Kategori']


/var/folders/9f/t69_qmyj3yl78q5qyr0y0bkr0000gn/T/ipykernel_32866/1207380718.py:132: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Saved → stacked_bar_Produkt_-_Produkt.png


KeyboardInterrupt: 